In [ ]:
%%bash
datasets download genome taxon "Staphylococcus aureus subsp. aureus NCTC 8325" \
  --assembly-source refseq \
  --assembly-level complete \
  --no-progressbar \
  --filename ../tmp/ncbi_dataset.zip \
  --include gbff

In [ ]:
from zipfile import ZipFile
from io import TextIOWrapper
from Bio import SeqIO
import pandas as pd

records = []
with ZipFile('../tmp/ncbi_dataset.zip') as z:
    gbff_files = [name for name in z.namelist() if name.endswith('.gbff')]
    print(f'Found {len(gbff_files)} GenBank files.')
    for gbff in gbff_files:
        with z.open(gbff) as raw:
            handle = TextIOWrapper(raw, encoding='utf-8')
            for rec in SeqIO.parse(handle, 'genbank'):
                records.append(rec)
                print(f'- {rec.id}\t{len(rec.seq):,} bp\t{rec.description[:80]} ...')

In [ ]:
genes = []
for record in records:
    for feature in record.features:
        if feature.type == "CDS":
            gene = feature.qualifiers.get("gene", [""])[0]
            if 'mdeA' in gene or 'mepA' in gene or 'nor' in gene:
                protein_id = feature.qualifiers.get("protein_id", [""])[0]
                product = feature.qualifiers.get("product", [""])[0]
                if protein_id and 'NP' not in protein_id and 'YP' not in protein_id:
                    genes.append([gene, protein_id, product])

if genes:
    genes = pd.DataFrame(genes).groupby([0,1,2], as_index=False).size().sort_values(by='size', ascending=False).groupby(0).first()

In [ ]:
for i, j in genes.iterrows():
    print(f'"{j[1]}": "{i}", // {j[2]}')

In [ ]:
genes[1].str.split('.').str.get(0).to_list()